# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmed-morad15/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [25]:
from google.colab import userdata
from huggingface_hub import login
import duckdb

# Load Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN was not found in Colab Secrets.")

print("HF_TOKEN loaded successfully.")

# Authenticate with Hugging Face
login(token=HF_TOKEN, add_to_git_credential=False)

# Start DuckDB
con = duckdb.connect()

print("DuckDB ready.")
print("Hugging Face authentication ready.")

HF_TOKEN loaded successfully.
DuckDB ready.
Hugging Face authentication ready.


In [26]:
REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("Warehouse relation ready.")

Warehouse relation ready.


In [27]:
REL_FEB = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
)
"""

REL_MAR = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("February and March warehouse relations ready.")

February and March warehouse relations ready.


In [28]:
con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("DuckDB Hugging Face secret configured.")

DuckDB Hugging Face secret configured.


In [29]:
baseline_df = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
    FROM {REL_FEB}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_avg_position) AS march_avg_position
    FROM {REL_MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    f.*,
    m.march_avg_position,

    CASE
        WHEN m.march_avg_position > f.gsc_avg_position
        THEN 1
        ELSE 0
    END AS review_priority

FROM feb f
INNER JOIN march m
    ON f.client_hash_id = m.client_hash_id
    AND f.content_hash_id = m.content_hash_id

WHERE f.gsc_avg_position IS NOT NULL
  AND m.march_avg_position IS NOT NULL
""").df()

print("Modeling dataset shape:", baseline_df.shape)
baseline_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling dataset shape: (134238, 9)


,client_hash_id,content_hash_id,gsc_avg_position,gsc_impressions,gsc_clicks,ga4_sessions,ga4_engaged_sessions,march_avg_position,review_priority
0,client_3ffa76342f366962,content_da44264c1fd25b4b,6.500000,11.0,0.0,0.0,0.0,6.214286,0
1,client_3ffa76342f366962,content_769436a447799cc7,9.338235,38.0,1.0,0.0,0.0,7.000000,0
2,client_3ffa76342f366962,content_32bdebcb01540202,3.815812,551.0,17.0,0.0,0.0,4.486420,1
3,client_3ffa76342f366962,content_18accd3f084fea96,3.818182,46.0,2.0,0.0,0.0,1.857143,0
4,client_3ffa76342f366962,content_63714a682809fe2a,6.594444,21.0,1.0,0.0,0.0,6.190909,0


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


I will start with Logistic Regression because this is a binary classification problem: predicting whether a content page should receive review priority. Logistic Regression provides a simple and interpretable learned baseline.

I will also train a Random Forest as a stronger non-linear comparison. It can capture interactions between the available GSC and GA4 signals without requiring a linear relationship.

The goal is not to prefer the more complex model automatically. I will keep the simpler method if it performs similarly, and only use the more complex model if the evaluation shows a meaningful improvement over the baseline.

The models will use only five February decision-time features:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_sessions`
- `ga4_engaged_sessions`

The target is `review_priority`, and March outcome information will not be used as a model feature.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


I will use a client-level holdout split. All content items belonging to the same client stay in the same split.

This is more honest than a random row split because multiple content items from the same client can share client-specific patterns. Keeping clients separate reduces the risk that the model learns client-specific information from training rows and is then evaluated on rows from the same client.

I will use 80% of clients for training and 20% for validation, with a fixed random seed for reproducibility.

The split is performed on `client_hash_id`, not individual rows. The validation set therefore contains clients that were not seen during training.

In [31]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

clients = baseline_df["client_hash_id"].dropna().unique()

train_clients, val_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_df = baseline_df[
    baseline_df["client_hash_id"].isin(train_clients)
].copy()

val_df = baseline_df[
    baseline_df["client_hash_id"].isin(val_clients)
].copy()

print("Total clients:", len(clients))
print("Train clients:", len(train_clients))
print("Validation clients:", len(val_clients))
print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))

Total clients: 42
Train clients: 33
Validation clients: 9
Train rows: 115625
Validation rows: 18613


In [32]:
train_client_set = set(train_df["client_hash_id"])
val_client_set = set(val_df["client_hash_id"])

overlap = train_client_set.intersection(val_client_set)

print("Client overlap:", len(overlap))

assert len(overlap) == 0

print("Client-level split check passed.")

Client overlap: 0
Client-level split check passed.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

FEATURES = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

TARGET = "review_priority"

X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].copy()

X_val = val_df[FEATURES].copy()
y_val = val_df[TARGET].copy()

print("Features:", FEATURES)
print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Positive rate:", y_val.mean())

Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions']
Training shape: (115625, 5)
Validation shape: (18613, 5)
Positive rate: 0.6181163702788374


In [35]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logreg = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

logreg.fit(X_train, y_train)

logreg_pred = logreg.predict(X_val)
logreg_prob = logreg.predict_proba(X_val)[:, 1]

print("Logistic Regression trained.")

Logistic Regression trained.


In [36]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_val)
rf_prob = rf.predict_proba(X_val)[:, 1]

print("Random Forest trained.")

Random Forest trained.


In [37]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()


# Reproduce the Week-4 baseline rule:
# pages with worse/slipping position receive higher priority.
baseline_scores = val_df["gsc_avg_position"].fillna(0).to_numpy()

# Model ranking scores
logreg_scores = logreg_prob
rf_scores = rf_prob

results = []

for name, scores in [
    ("Baseline", baseline_scores),
    ("Logistic Regression", logreg_scores),
    ("Random Forest", rf_scores),
]:
    results.append({
        "model": name,
        "base_rate": y_val.mean(),
        "precision_at_10": precision_at_k(scores, y_val, 10),
        "precision_at_50": precision_at_k(scores, y_val, 50),
    })

comparison = pd.DataFrame(results)

display(comparison)

,model,base_rate,precision_at_10,precision_at_50
0,Baseline,0.618116,0.0,0.02
1,Logistic Regression,0.618116,0.7,0.68
2,Random Forest,0.618116,0.8,0.82


In [38]:
classification_results = []

for name, predictions in [
    ("Logistic Regression", logreg_pred),
    ("Random Forest", rf_pred),
]:
    classification_results.append({
        "model": name,
        "accuracy": accuracy_score(y_val, predictions),
        "precision": precision_score(y_val, predictions, zero_division=0),
        "recall": recall_score(y_val, predictions, zero_division=0),
        "f1": f1_score(y_val, predictions, zero_division=0),
    })

classification_comparison = pd.DataFrame(classification_results)

classification_comparison

,model,accuracy,precision,recall,f1
0,Logistic Regression,0.629936,0.627253,0.989048,0.767658
1,Random Forest,0.605545,0.659221,0.749066,0.701278


In [39]:
print("Base rate:", round(y_val.mean(), 4))
print("\nRanking comparison:")
display(comparison)

print("\nClassification comparison:")
display(classification_comparison)

Base rate: 0.6181

Ranking comparison:


,model,base_rate,precision_at_10,precision_at_50
0,Baseline,0.618116,0.0,0.02
1,Logistic Regression,0.618116,0.7,0.68
2,Random Forest,0.618116,0.8,0.82



Classification comparison:


,model,accuracy,precision,recall,f1
0,Logistic Regression,0.629936,0.627253,0.989048,0.767658
1,Random Forest,0.605545,0.659221,0.749066,0.701278


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*


The Random Forest relies most strongly on `gsc_avg_position` (0.603) and `gsc_impressions` (0.309), with smaller contributions from `gsc_clicks`, `ga4_sessions`, and `ga4_engaged_sessions`. This is consistent with the observed relationship between search visibility and review priority, but these importances are directional and do not imply causation.

The model made 7,902 errors on the client-held-out validation set. The inspected errors include both false positives and false negatives, especially around intermediate search positions where the available signals do not cleanly separate the two classes.

For example, one false positive had position 4.0 with low impressions, while another false positive had position 12.33. A false negative had position 12.43 despite being a positive case. These examples show that position alone is not sufficient to identify every review-priority page.


In [41]:
# Get the Random Forest estimator from the pipeline
rf_model = rf.named_steps["model"]

# Feature importance
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

print("Random Forest feature importance:")
display(feature_importance)

# Validation predictions
rf_pred = rf.predict(X_val)
rf_prob = rf.predict_proba(X_val)[:, 1]

# Find incorrect predictions
error_mask = rf_pred != y_val

errors = X_val.loc[error_mask].copy()
errors["actual"] = y_val.loc[error_mask]
errors["predicted"] = rf_pred[error_mask]
errors["predicted_probability"] = rf_prob[error_mask]

print(f"Validation errors: {len(errors)}")
print("\nThree concrete validation errors:")
display(errors.head(3))

Random Forest feature importance:


,feature,importance
2,gsc_avg_position,0.600578
0,gsc_impressions,0.311091
1,gsc_clicks,0.050885
3,ga4_sessions,0.030624
4,ga4_engaged_sessions,0.006822


Validation errors: 7342

Three concrete validation errors:


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,actual,predicted,predicted_probability
3365,13.0,0.0,8.725000,0.0,0.0,1,0,0.085000
3366,1.0,0.0,10.000000,0.0,0.0,1,0,0.383037
3367,24.0,0.0,9.630952,0.0,0.0,1,0,0.240000


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.